# SIRCA — Equipe E — Ciclo 2
Execute em ordem. O código é público; a base deve ser usada apenas no ambiente autorizado pelo professor. Este notebook não contém dados do caso. Não salve nem compartilhe saídas com registros pessoais. Apoio de ChatGPT/Codex declarado; a equipe deve executar e revisar.


In [ ]:
from pathlib import Path
import subprocess, sys, shutil, tempfile, zipfile
checkout = Path('/content/RIAN-1PROJETO')
if not checkout.exists():
    subprocess.run(['git','clone','https://github.com/gomesrian241-tech/RIAN-1PROJETO.git',str(checkout)],check=True)
RAIZ = checkout / 'sirca'
%cd /content/RIAN-1PROJETO/sirca
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)


## Testes sem a base
49 testes podem ser executados antes do envio dos dados. Os dois testes de integração serão executados depois da carga.


In [ ]:
subprocess.run([sys.executable,'-m','pytest','componente/testes','-q','-k','not test_csv_igual_modelo_etl and not test_carga_balanco_sem_descartes_ocultos'],check=True)


## Enviar a base autorizada
Escolha os três arquivos originais juntos ou o ZIP técnico `SIRCA_EQUIPE_E_CICLO_2.zip`. O código extrai somente os três brutos esperados, sem descompactar outros caminhos. Use o ambiente autorizado; estes arquivos ficam temporariamente na sessão. Nenhum conteúdo é exibido nesta etapa.


In [ ]:
from google.colab import files
esperados = {'Instituto_Semear_planilhas_operacionais_2024-2026.xlsx','frequencia_export_sistema_antigo.csv','doacoes_2026.csv'}
brutos = RAIZ / 'dados/brutos'
brutos.mkdir(parents=True,exist_ok=True)
with tempfile.TemporaryDirectory() as temporario:
    import os
    anterior = Path.cwd()
    try:
        os.chdir(temporario)
        enviados = files.upload()
        encontrados = {}
        for nome,conteudo in enviados.items():
            if Path(nome).name in esperados:
                chave = Path(nome).name
                if chave in encontrados: raise ValueError('Arquivo duplicado: '+chave)
                encontrados[chave] = conteudo
            elif nome.lower().endswith('.zip'):
                import io
                with zipfile.ZipFile(io.BytesIO(conteudo)) as pacote:
                    for item in pacote.infolist():
                        chave = Path(item.filename).name
                        if chave in esperados:
                            if chave in encontrados: raise ValueError('Arquivo duplicado: '+chave)
                            encontrados[chave] = pacote.read(item)
        faltantes = esperados - encontrados.keys()
        if faltantes: raise ValueError('Faltam: '+', '.join(sorted(faltantes)))
        for nome in esperados:
            (brutos/nome).write_bytes(encontrados[nome])
    finally:
        os.chdir(anterior)
print('Três arquivos disponíveis para tratamento; conteúdo não exibido.')


## Tratar, testar e conferir
A carga gera relatório de rejeições e conversões de presença, preservando a origem de cada decisão. As células seguintes podem gerar saídas derivadas da base; revise o compartilhamento do notebook antes de salvar resultados.


In [ ]:
subprocess.run([sys.executable,'scripts/tratar_dados.py'],check=True)
subprocess.run([sys.executable,'-m','pytest','componente/testes','-q'],check=True)
subprocess.run([sys.executable,'componente/src/main.py','--resumo'],check=True)
(RAIZ/'banco/semear_dump.sql').write_text((RAIZ/'banco/01_criar_tabelas.sql').read_text()+'\n'+(RAIZ/'banco/02_carregar_dados.sql').read_text())


In [ ]:
import shutil
if shutil.which('node') is None or shutil.which('npm') is None:
    raise RuntimeError('Node e npm são necessários para conferir PostgreSQL embarcado; use também as instruções locais do README.')
subprocess.run(['npm','install'],check=True)
subprocess.run(['npm','run','test:banco'],check=True)
subprocess.run([sys.executable,'scripts/conferir_resultados.py'],check=True)
subprocess.run([sys.executable,'estudo/calcular.py'],check=True)


## Usar o componente
O menu é interativo. Para encerrar, use a opção de saída. Exporte os lançamentos da sessão pelo menu antes de encerrar, se precisar preservá-los.


In [ ]:
%run componente/src/main.py


## Evidências e entrega
Os resultados ficam em `dados/tratados/`, `evidencias/` e `banco/resultados/` nesta sessão. Salve-os somente no local restrito autorizado. A planilha preenchida e o PDF revisado estão no pacote técnico da equipe, não são recriados por este notebook. Atualize no PDF os links e caminhos de cada material. O notebook não substitui atas, identificação, assinaturas nem evidências individuais.
